# `basin` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'basin'
feature_metadata = {'order': 10, 'name': 'basin', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain as stable low-cardinality geography', 'finding': 'Nine complete levels have stable train/test coverage and meaningful target differences.', 'decision': 'Retain as a categorical feature and compare its contribution with administrative geography.', 'risk': 'Hydrological and administrative geography overlap without forming a strict hierarchy.', 'related': [{'feature': 'region', 'reason': 'Hydrological basins cross administrative regions.'}, {'feature': 'source', 'reason': 'Water source types differ across hydrological basins.'}, {'feature': 'longitude', 'reason': 'Basin assignment is spatially structured.'}, {'feature': 'latitude', 'reason': 'Basin assignment is spatially structured.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for basin.


## Supported target evidence


In [2]:
sentinel_tokens = []
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
basin,,,,,
lake victoria,10248,True,49.77,9.65,40.58
pangani,8940,True,60.09,5.34,34.57
rufiji,7976,True,63.54,5.48,30.98
internal,7785,True,57.57,7.15,35.27
lake tanganyika,6432,True,48.31,11.54,40.16
wami / ruvu,5987,True,52.38,4.49,43.13
lake nyasa,5085,True,65.37,4.92,29.71
ruvuma / southern coast,4493,True,37.17,7.26,55.58
lake rukwa,2454,True,40.75,11.00,48.25


status_group,rows,non functional (%)
basin,,
ruvuma / southern coast,4493,55.58
lake rukwa,2454,48.25
wami / ruvu,5987,43.13
lake victoria,10248,40.58
lake tanganyika,6432,40.16
internal,7785,35.27
pangani,8940,34.57
rufiji,7976,30.98
lake nyasa,5085,29.71


## Observation

Nine complete levels have stable train/test coverage and meaningful target differences.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Retain as a categorical feature and compare its contribution with administrative geography.

**Risk to carry forward:** Hydrological and administrative geography overlap without forming a strict hierarchy.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
basin,candidate,retain as stable low-cardinality geography,Nine complete levels have stable train/test co...,Retain as a categorical feature and compare it...,Hydrological and administrative geography over...
